# NewsAPI vs Factiva

This Jupyter Notebook compares query results for Mashreq countries of NewsAPI (https://newsapi.ai/) and Factiva. 


Author: Dominik Wielath (dominik.wielath@gmail.com)

DIME Artificial Intelligence (DIME AI) - World Bank Group

Date: 2024-07-10

In [1]:
from eventregistry import *
import pandas as pd
import numpy as np
from datetime import datetime

import os
import re

from utilities import *

In [2]:
mashreq_countries = ["Bahrain", "Egypt", "Iraq", "Israel", "Jordan", "Kuwait", "Lebanon", "Libya", "Oman", "Qatar", "Saudi Arabia", "Sudan", "Palestine", "Syria", "United Arab Emirates", "Yemen"]

## 1. Top Sources per country from 06/16/2024 to 06/30/2024

In [3]:
# Set start and end dates
start_date_string = "06/16/2024"
end_date_string = "06/30/2024"

date_format = "%m/%d/%Y"
start_date = datetime.strptime(start_date_string, date_format)
end_date = datetime.strptime(end_date_string, date_format)

# Create date format for queries as required by News API
query_date_format = "%Y-%m-%d"
start_date_query_format = start_date.strftime(query_date_format)
end_date_query_format = end_date.strftime(query_date_format)

# Create date format for filenames
filename_date_format = "%m%d%Y"
start_date_filename_format = start_date.strftime(filename_date_format)
end_date_filename_format = end_date.strftime(filename_date_format)

In [4]:
# Set language for articles in query
language = "eng"

### 1.1 Create NewsAPI queries and save query results

In [16]:
data_path_newsapi = "../../data/newsapi/"

In [17]:
News_API_key = '7ce3fa46-3bab-4460-a552-e3046d565492'
er = EventRegistry(apiKey = News_API_key)

In [72]:
query_string = """{
  "$query": {
    "$and": [
      {
            "conceptUri": "%s"
      },
      {
        "dateStart": "%s",
        "dateEnd": "%s",
        "lang": "%s"
      }
    ]
  },
  "$filter": {
    "isDuplicate": "skipDuplicates"
  }
}"""

In [74]:
for country in mashreq_countries:    
    query = query_string % (er.getConceptUri(country), start_date_query_format, end_date_query_format, language)
    q = QueryArticlesIter.initWithComplexQuery(query)
    q.setRequestedResult(RequestArticlesSourceAggr(sourceCount = 100))
    res = er.execQuery(q)

    dict_list = []
    for i in range(len(res["sourceAggr"]["countsPerSource"])):
        dict_list.append(res["sourceAggr"]["countsPerSource"][i]["source"] | res["sourceAggr"]["countsPerSource"][i]["counts"])
        
    region = country.lower().replace(" ", "_")
    no_results = res["sourceAggr"]["totalResults"]
    file_name_newsapi = f"{region}_{from_date}_{to_date}_{language}_{no_results}_newsapi.csv"
    pd.DataFrame(dict_list).to_csv(data_path_newsapi + file_name, index = False)
    print(f"{country}: {file_name_newsapi} saved successfully!")

Bahrain: bahrain_06162024_06302024_eng_1885_newsapi.csv saved successfully!
Egypt: egypt_06162024_06302024_eng_10069_newsapi.csv saved successfully!
Iraq: iraq_06162024_06302024_eng_6490_newsapi.csv saved successfully!
Israel: israel_06162024_06302024_eng_33084_newsapi.csv saved successfully!
Jordan: jordan_06162024_06302024_eng_7500_newsapi.csv saved successfully!
Kuwait: kuwait_06162024_06302024_eng_1936_newsapi.csv saved successfully!
Lebanon: lebanon_06162024_06302024_eng_7893_newsapi.csv saved successfully!
Libya: libya_06162024_06302024_eng_1248_newsapi.csv saved successfully!
Oman: oman_06162024_06302024_eng_2096_newsapi.csv saved successfully!
Qatar: qatar_06162024_06302024_eng_5940_newsapi.csv saved successfully!
Saudi Arabia: saudi_arabia_06162024_06302024_eng_13555_newsapi.csv saved successfully!
Sudan: sudan_06162024_06302024_eng_2683_newsapi.csv saved successfully!
Palestine: palestine_06162024_06302024_eng_1949_newsapi.csv saved successfully!
Syria: syria_06162024_0630202

### 1.2 Clean Factiva top sources results and save dataframes

In [5]:
data_path_raw = "../../data/factiva/raw/"
data_path_clean = "../../data/factiva/clean/"

In [7]:
clean_factiva_most_mentioned_sources_expoert(data_path_raw, data_path_clean)

Cleaning file: Source(14).csv
File saved: Syria_06162024_06302024_eng_1085.csv

Cleaning file: Source(6).csv
File saved: Kuwait_06162024_06302024_eng_1872.csv

Cleaning file: Source(12).csv
File saved: Sudan_06162024_06302024_eng_1552.csv

Cleaning file: Source(1).csv
File saved: Bahrain_06162024_06302024_eng_969.csv

Cleaning file: Source(2).csv
File saved: Egypt_06162024_06302024_eng_4571.csv

Cleaning file: Source(11).csv
File saved: Saudi_Arabia_06162024_06302024_eng_6897.csv

Cleaning file: Source(15).csv
File saved: United_Arab_Emirates_06162024_06302024_eng_7232.csv

Cleaning file: Source(3).csv
File saved: Iraq_06162024_06302024_eng_2374.csv

Cleaning file: Source(9).csv
File saved: Oman_06162024_06302024_eng_1231.csv

Cleaning file: Source(7).csv
File saved: Lebanon_06162024_06302024_eng_5597.csv

Cleaning file: Source(4).csv
File saved: Israel_06162024_06302024_eng_26193.csv

Cleaning file: Source(16).csv
File saved: Yemen_06162024_06302024_eng_2642.csv

Cleaning file: Source